# Phương pháp thu thập dữ liệu bài báo khoa học

> **Môn học:** Nhập môn Khoa học Dữ liệu — HCMUS 
> **Milestone 1:** Thu thập Dữ liệu và Đo lường Hiệu năng 
> **Nhóm thực hiện:** Nhóm 04 (Đặng Ngọc Hoàng, Dương Ngọc Minh Thư, Trần Ngọc Thanh Tú, Cáp Hửu Duy, Nguyễn Lê Anh Tuấn) 
> **Thời gian thực hiện:** Học kỳ 1, Năm học 2026–2027

---

## 1. Mục tiêu và phạm vi thu thập

Mục tiêu của Milestone 1 là xây dựng một quy trình thu thập dữ liệu khoa học tự động, mạnh mẽ và có thể tái lập, nhằm chuẩn bị tập dữ liệu bài báo phục vụ cho bài toán khai phá dữ liệu văn bản và trích xuất mạng lưới trích dẫn (Milestone 2 & 3).

### Yêu cầu cốt lõi theo đề bài:
1. **Metadata bài báo:** Thu thập đầy đủ các trường thông tin theo bảng `papers` (arXiv ID, Title, Authors, Submission Date, Revised Dates, Venue).
2. **Toàn văn mã nguồn TeX:** Tải về toàn bộ các version có sẵn của từng bài báo từ arXiv mirror (`export.arxiv.org/src/`), giải nén và giữ lại duy nhất các file `.tex` và `.bib` (loại bỏ hoàn toàn các file ảnh, PDF, và file nhị phân trung gian).
3. **Mạng lưới trích dẫn:** Khai thác danh mục tài liệu tham khảo (references) từ Semantic Scholar Graph API cho từng bài báo tương ứng.
4. **Đo lường hiệu năng:** Ghi nhận có hệ thống 4 chỉ số tài nguyên: End-to-end wall-clock time, Peak RAM, Average RAM và Disk usage delta.


---

## 2. Khảo sát và so sánh các nguồn dữ liệu

Để thu thập kho dữ liệu bài báo khoa học quy mô lớn, nhóm đã khảo sát 4 phương thức và nguồn cung cấp dữ liệu:

1. **arXiv API (Atom Feed 1.0):** Giao thức HTTP GET chính thức do arXiv cung cấp.
2. **OAI-PMH :** Giao thức thu hoạch siêu dữ liệu hàng loạt theo chuẩn Dublin Core và arXiv Raw XML.
3. **Kaggle arXiv Dataset:** Bộ dữ liệu tổng hợp đóng gói sẵn dạng snapshot trên Kaggle.
4. **Semantic Scholar Graph API:** Nền tảng đồ thị học thuật phục vụ khai thác trích dẫn sâu.

### So sánh đặc tính kỹ thuật giữa các nguồn dữ liệu:

| Tiêu chí Đánh giá | arXiv API | OAI-PMH (Endpoint Mới) | Kaggle arXiv Dataset | Semantic Scholar API |
|---|---|---|---|---|
| **Giao thức truyền tải** | HTTP REST (Atom 1.0) | HTTP OAI-PMH (XML) | File Download | HTTP REST (JSON) |
| **Giới hạn tốc độ truy cập** | 1 request / 3 giây | 1 request / 3 giây | Không giới hạn (Tải một lần) | 1 request / giây (Miễn phí) |
| **Độ trễ cập nhật** | Thời gian thực (thời gian thực) | Thời gian thực (thời gian thực) | Snapshot tĩnh (Trễ 1–3 tháng) | Trễ từ 1–7 ngày |
| **Phạm vi dữ liệu** | Siêu dữ liệu bài báo | Toàn bộ lịch sử metadata | Siêu dữ liệu + Abstract | Mạng lưới trích dẫn |
| **Hỗ trợ truy vấn theo ID** | Hỗ trợ danh sách ID | Truy vấn tuần tự qua token | Cần duyệt toàn bộ file lớn | Hỗ trợ batch tối đa 500 ID |
| **Vai trò trong đồ án** | **Nguồn chính ** | Nguồn dự phòng kiểm định | Đối sánh cross-check | **Nguồn chính cho References** |


---

## 3. Phân tích chi tiết từng nguồn dữ liệu

### 3.1. Giao diện lập trình arXiv API
- **Cơ chế hoạt động:** Truy vấn qua HTTP GET với tham số `id_list` hoặc `search_query` đến endpoint `http://export.arxiv.org/api/query`. Kết quả trả về dưới dạng Atom XML.
- **Thư viện tương tác `arxiv.py` & Những thay đổi lớn (Breaking Changes 2025–2026):**
 * Thư viện `arxiv.py` đã loại bỏ hoàn toàn cú pháp cũ `search.results()` và các phương thức tải trực tiếp `Result.download_pdf()` / `Result.download_source()` trong phiên bản 4.0.0 (tháng 05/2026).
 * Quy chuẩn hiện đại bắt buộc sử dụng client hướng đối tượng: `arxiv.Client().results(search)`.
 * Do thư viện không còn hỗ trợ tải mã nguồn trực tiếp, toàn bộ source archives được tải trực tiếp qua HTTP stream từ dedicated mirror `https://export.arxiv.org/src/{arxiv_id}` (hoặc bản có version `.../src/{arxiv_id}v{version}`).
- **Hạn chế kỹ thuật quan trọng:**
 > **Hạn chế về `revised_dates`:** arXiv Atom feed chỉ cung cấp 2 mốc thời gian: `published` (ngày nộp bản đầu tiên $v_1$) và `updated` (ngày cập nhật bản mới nhất). Đối với các bài báo có từ 3 version trở lên ($n \ge 3$), ngày của các version trung gian ($v_2, v_3, \dots, v_{n-1}$) **hoàn toàn không có trong phản hồi Atom**.
 > Nhóm ghi nhận trường hợp này với cờ `"revised_dates_complete": false` để bảo đảm tính minh bạch và truy xuất nguồn gốc dữ liệu.

### 3.2. Giao thức thu hoạch siêu dữ liệu OAI-PMH
- **Cơ chế hoạt động:** Sử dụng các động từ (verbs) chuẩn của giao thức OAI-PMH: `ListRecords`, `GetRecord`, `Identify`, `ListSets`.
- **Cập nhật quan trọng về Endpoint (Tháng 03/2025):** 
 * Endpoint cũ `export.arxiv.org/oai2` đã chính thức ngừng hoạt động trong đợt chuyển đổi hạ tầng đám mây của arXiv.
 * Toàn bộ truy vấn mới bắt buộc hướng về endpoint chính thức: `https://oaipmh.arxiv.org/oai`.
- **Đặc điểm của Resumption Token:**
 * Kể từ đợt nâng cấp tháng 03/2025, resumption token không còn mã hóa tổng số lượng bản ghi (collection size) và tự động hết hạn hàng ngày. Do đó, các tiến trình thu hoạch quy mô lớn cần được thiết kế hoàn tất trong ngày hoặc lưu checkpoint trạng thái để khôi phục linh hoạt.
 * Tốc độ truy vấn vẫn phải tuân thủ nghiêm ngặt pacing $\ge 3.0$ giây giữa các request liên tiếp.
- **Giá trị khai thác:** Metadata định dạng `arXivRaw` trong OAI-PMH chứa lịch sử chi tiết (full version history) bao gồm kích thước và ngày nộp chính xác của từng version $v_1, v_2, \dots, v_n$ (được khai thác ở Phân hệ A3 của Đặng Ngọc Hoàng).

### 3.3. Tập dữ liệu ảnh chụp Kaggle arXiv
- **Cơ chế hoạt động:** Bản snapshot tĩnh dạng JSON lines (cung cấp bởi Đại học Cornell trên Kaggle), dung lượng khoảng 4GB nén (14GB giải nén), chứa hơn 2 triệu bài báo.
- **Giá trị khai thác:** Cho phép đọc và lọc siêu dữ liệu cục bộ bằng `pandas` mà không cần gọi API, lý tưởng để đối chiếu chéo (cross-validation) dữ liệu độc lập với máy chủ arXiv.

### 3.4. Giao diện lập trình đồ thị Semantic Scholar API
- **Cơ chế hoạt động:** Truy vấn REST API qua mã định danh `arXiv:<id>`.
 - Endpoint: `https://api.semanticscholar.org/graph/v1/paper/arXiv:{arxiv_id}`
- **Quy tắc tham số phẳng :** API Semantic Scholar yêu cầu các trường khai báo ở dạng phẳng (ví dụ: `fields=references.title,references.authors,references.year,references.publicationDate,references.venue,references.externalIds`). Các cú pháp lồng nhau như `authors.name` sẽ bị máy chủ từ chối với lỗi HTTP 400.
- **Xử lý mã định danh arXiv cũ và trích dẫn ngoài:**
 * Mỗi tài liệu tham khảo trả về từ API đều chứa `paperId` (định danh Semantic Scholar) và từ điển `externalIds`.
 * Các bài báo arXiv xuất bản trước năm 2007 sử dụng định dạng cũ dạng `archive/YYMMNNN` (ví dụ: `hep-th/9901001`). Hệ thống giữ nguyên vẹn các định danh này để đảm bảo tính liên kết.
 * Các trích dẫn không thuộc arXiv (sách, hội thảo ngoài, báo cáo kỹ thuật) vẫn được lưu trữ đầy đủ trong file `references.json` cục bộ, và chỉ những trích dẫn có `externalIds.ArXiv` mới được đưa vào bảng `references` chính thức cho Milestone 3.

### 3.5. Phương pháp đối sánh chéo ba nguồn siêu dữ liệu
Để đánh giá độ tin cậy và phát hiện các khiếm khuyết tiềm ẩn của từng nguồn dữ liệu, nhóm trưởng Đặng Ngọc Hoàng đã thiết kế quy trình đối sánh chéo 3 chiều (3-Way Cross-Validation) giữa arXiv API, OAI-PMH và Kaggle trên 5.000 bài báo. Kết quả chi tiết được phân tích tại Mục 5 của tài liệu [03-performance.ipynb](03-performance.ipynb).

---

## 4. Chuẩn hóa cấu trúc và đặc tả các bảng dữ liệu xuất bản

Để đảm bảo tính nhất quán xuyên suốt đồ án, nhóm đã thống nhất hợp đồng dữ liệu chuẩn tại `docs/schema.md`.

### 4.1. Quy ước khóa định danh bài báo và tổ chức thư mục:
- **Tổ chức thư mục làm việc:** Mỗi bài báo được lưu trong một thư mục riêng biệt với tên thư mục an toàn `pub_key` (thay dấu `.` thành `-` và `/` thành `_`, ví dụ `2504-07523` hoặc `hep-th_9901001`). Thư mục này chứa:
 * Toàn bộ mã nguồn `.tex` và `.bib` của từng version ($v_1, v_2, \dots$).
 * Siêu dữ liệu bài báo `paper_meta.json`.
 * Danh mục trích dẫn đầy đủ từ Semantic Scholar `references.json`.
 * File đánh dấu `_EMPTY.json` đối với các version không có mã nguồn TeX (chỉ có PDF/Word).

### 4.2. Ánh xạ trường thông tin vào bảng dữ liệu `papers`:

| Tên trường Chuẩn | Kiểu dữ liệu | Nguồn arXiv API / OAI-PMH | Nguồn Semantic Scholar | Ghi chú xử lý |
|---|---|---|---|---|
| `arxiv_id` | String (bắt buộc) | `id` (chuẩn hóa loại bỏ URL) | `externalIds.ArXiv` | Khóa chính của bài báo |
| `title` | String | `title` (loại bỏ xuống dòng thừa) | `title` | Chuẩn hóa khoảng trắng |
| `authors` | List[String] | Danh sách thẻ `author/name` | `authors[].name` | Chuẩn hóa danh sách tên tác giả |
| `abstract` | String | `summary` | `abstract` | Giữ nguyên nội dung tóm tắt |
| `submission_date`| String (ISO 8601) | `published` | `publicationDate` | Mốc thời gian nộp bản đầu tiên ($v_1$) |
| `revised_dates` | List[String] | Thẻ `published` (v1) và `updated` (bản mới nhất) trên API; các version trung gian ($n \ge 3$) lấy bổ sung từ OAI-PMH (`arXivRaw`) hoặc Kaggle arXiv Dataset | Không có | Lịch sử cập nhật các phiên bản |
| `venue` | String (tùy chọn) | Thẻ `arxiv:journal_ref` / `comment`| `venue` | Tạp chí hoặc hội nghị công bố |
| `primary_category`| String | `arxiv:primary_category/@term`| Không có | Phân loại chính (ví dụ: `cs.LG`) |
| `categories` | List[String] | Danh sách thẻ `category/@term` | `fieldsOfStudy` | Toàn bộ chuyên ngành phân loại |

> **Ghi chú chuẩn hóa bảng `papers`:** Theo đúng đặc tả đề bài, bảng `papers` xuất bản gồm đúng **6 trường chuẩn**: `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue`. Các trường bổ trợ như `abstract`, `primary_category`, `categories` được lưu trữ nội bộ trong `paper_meta.json` để phục vụ các phân tích mở rộng của Milestone 2.

### 4.3. Đặc tả hai bảng dữ liệu xuất bản phục vụ giai đoạn 3

Theo yêu cầu chuẩn mực của Milestone 1, dữ liệu thu thập được thiết kế để trực tiếp chuyển hóa thành 2 bảng dữ liệu xuất bản chính thức:

#### Bảng 1: `papers`
| Tên cột | Kiểu dữ liệu | Ràng buộc | Diễn giải nghiệp vụ |
|---|---|---|---|
| `arxiv_id` | String | Non-null | Khóa chính, mã định danh bài báo (ví dụ: `2504.07523`) |
| `title` | String | Non-null | Tiêu đề đầy đủ của bài báo |
| `authors` | List[String] | Non-null | Danh sách các tác giả bài báo |
| `submission_date` | String (ISO 8601) | Non-null | Ngày nộp phiên bản đầu tiên ($v_1$) dạng `YYYY-MM-DDTHH:MM:SSZ` |
| `revised_dates` | List[String] | Nullable | Danh sách các mốc thời gian cập nhật phiên bản tiếp theo |
| `venue` | String | Nullable | Nơi xuất bản chính thức (Journal / Conference / Workshop) |

#### Bảng 2: `references`
| Tên cột | Kiểu dữ liệu | Ràng buộc | Diễn giải nghiệp vụ |
|---|---|---|---|
| `arxiv_id` | String | Non-null | Mã định danh của bài báo trích dẫn |
| `ref_arxiv_id` | String | Non-null | Mã định danh arXiv của bài được trích dẫn |
| `ref_title` | String | Nullable | Tiêu đề của bài báo được trích dẫn |
| `ref_authors` | List[String] | Nullable | Danh sách tác giả của bài báo được trích dẫn |
| `ref_submission_date`| String | Nullable | Ngày công bố của bài báo được trích dẫn |
| `ref_s2_id` | String | Non-null | Mã định danh Paper ID của Semantic Scholar |

> **Quy tắc Matching Pool cho Milestone 3:** Chỉ các tài liệu tham khảo có chứa `ref_arxiv_id` hợp lệ mới trở thành một dòng trong bảng `references` xuất bản, phục vụ trực tiếp cho bài toán liên kết mạng lưới trích dẫn.


---

## 5. Ràng buộc giới hạn tốc độ và giải thuật kiểm soát lỗi

### 5.1. Ràng buộc từ phía máy chủ dịch vụ:
- **Quy định của arXiv:** Bắt buộc chờ tối thiểu **3.0 giây** giữa 2 yêu cầu liên tiếp và chỉ mở **1 kết nối đồng thời** (`REQUEST_DELAY = 3.0s`). Đồng thời, bắt buộc sử dụng header `User-Agent` mang định danh rõ ràng theo quy định:

```text
User-Agent: <GroupName>/1.0 (academic course project; mailto:<email>)
```

- **Quy định của Semantic Scholar:** Giới hạn tối đa **1 request/giây** đối với tài khoản miễn phí (`S2_REQUEST_DELAY = 1.1s`), và hỗ trợ gửi batch tối đa 500 ID mỗi request khi sử dụng API key.

### 5.2. Giải thuật chờ lũy thừa kết hợp nhiễu ngẫu nhiên:
Khi gặp mã lỗi HTTP `429 ` hoặc các lỗi máy chủ tạm thời `5xx`, hệ thống áp dụng giải thuật chờ lũy thừa kết hợp nhiễu ngẫu nhiên để tránh xung đột đồng thời:

```text
t_wait = random(0, min(t_max, t_base * 2^retry))
```

Dưới đây là mã nguồn minh họa giải thuật xử lý retry trong pipeline:


In [1]:
import time
import random
import requests

def fetch_with_backoff(url: str, headers: dict, max_retries: int = 4, base_delay: float = 3.0):
    """Truy vấn HTTP với cơ chế Exponential Backoff with Jitter."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            if resp.status_code == 200:
                return resp
            elif resp.status_code in (429, 500, 502, 503, 504):
                cap = min(60.0, base_delay * (2 ** attempt))
                sleep_time = random.uniform(base_delay, cap)
                print(f"[Retry {attempt+1}/{max_retries}] HTTP {resp.status_code}. Backing off for {sleep_time:.2f}s...")
                time.sleep(sleep_time)
            else:
                resp.raise_for_status()
        except requests.RequestException as exc:
            if attempt == max_retries - 1:
                raise
            sleep_time = random.uniform(base_delay, base_delay * (2 ** attempt))
            print(f"[Retry {attempt+1}/{max_retries}] Lỗi kết nối: {exc}. Chờ {sleep_time:.2f}s...")
            time.sleep(sleep_time)
    raise RuntimeError(f"Vượt quá số lần retry cho URL: {url}")

# Thực nghiệm mô phỏng khoảng thời gian chờ Exponential Backoff with Jitter (Base = 3.0s, Max = 60.0s)
print("MÔ PHỎNG GIẢI THUẬT EXPONENTIAL BACKOFF WITH FULL JITTER")
print("______________________________________________________________________________")
random.seed(42)
base_delay = 3.0
for attempt in range(4):
    cap = min(60.0, base_delay * (2 ** attempt))
    sample_delays = [random.uniform(base_delay, cap) for _ in range(3)]
    samples_str = ", ".join(f"{d:.2f}s" for d in sample_delays)
    print(f"Lần thử {attempt+1}: Cận trên cap = {cap:4.1f}s | Mẫu thực nghiệm: [{samples_str}]")


MÔ PHỎNG GIẢI THUẬT EXPONENTIAL BACKOFF WITH FULL JITTER
______________________________________________________________________________
Lần thử 1: Cận trên cap =  3.0s | Mẫu thực nghiệm: [3.00s, 3.00s, 3.00s]
Lần thử 2: Cận trên cap =  6.0s | Mẫu thực nghiệm: [3.67s, 5.21s, 5.03s]
Lần thử 3: Cận trên cap = 12.0s | Mẫu thực nghiệm: [11.03s, 3.78s, 6.80s]
Lần thử 4: Cận trên cap = 24.0s | Mẫu thực nghiệm: [3.63s, 7.59s, 13.61s]


---

## 6. Xử lý toàn văn mã nguồn TeX và tối ưu dung lượng lưu trữ

Đề bài yêu cầu tải toàn văn mã nguồn của **tất cả các version có sẵn** và giải nén lấy nội dung `.tex` và `.bib`. Thách thức kỹ thuật lớn nhất ở bước này là dung lượng đĩa: trung bình một file `.tar.gz` chứa đầy đủ hình ảnh, biểu đồ độ phân giải cao và file trung gian có thể nặng tới 10–50 MB, khiến kho lưu trữ 5.000 bài có thể chạm ngưỡng 30–35 GB (và toàn bộ 25.000 bài của cả nhóm lên tới hơn 166 GB archive nén thô).

### 6.1. Quy trình xử lý gói nén mã nguồn:
1. **Tải về có kiểm soát:** Đọc luồng dữ liệu nhị phân trực tiếp từ HTTP response, kiểm tra Magic Bytes để nhận diện định dạng (gzip, tar, hay PDF thuần).
2. **Lọc bỏ dữ liệu nhị phân :** Chỉ trích xuất các tệp có phần mở rộng `.tex` và `.bib`. Toàn bộ file ảnh (`.png`, `.jpg`, `.pdf`, `.eps`), file định dạng (`.sty`, `.cls`) và file biên dịch (`.bbl`, `.aux`) đều bị loại bỏ.
3. **Xử lý bản ghi rỗng (_EMPTY.json):** Nếu bài báo không có mã nguồn TeX (tác giả chỉ nộp PDF), version đó được đánh dấu bằng file `_EMPTY.json` theo đúng quy ước schema.


---

## 7. Khả năng tái lập và hạ tầng ghi nhận nhật ký

Mọi hoạt động trong pipeline đều tuân thủ nguyên tắc **Idempotence (Tính bất biến khi chạy lại)**:
- Khi chạy lại lệnh cào hoặc giải nén, hệ thống tự động kiểm tra sự tồn tại của file đầu ra; nếu đã có và hợp lệ thì bỏ qua, không gửi request trùng lặp lên máy chủ.
- Mỗi lần thực thi đều ghi nhận nhật ký chi tiết vào các tệp CSV chuyên biệt (`logs/discovery.csv`, `logs/download.csv`, `logs/extract.csv`, `logs/references.csv`, `logs/benchmark_runs.csv`), phục vụ việc truy vết lỗi và phân tích thống kê định lượng.

## 8. Tài liệu tham khảo chính thức
1. [arXiv API User Manual](https://info.arxiv.org/help/api/user-manual.html) — Hướng dẫn kỹ thuật và quy tắc etiquette.
2. [Open Archives Initiative Protocol for Metadata Harvesting (OAI-PMH v2.0)](https://www.openarchives.org/OAI/openarchivesprotocol.html).
3. [Semantic Scholar Academic Graph API Documentation](https://api.semanticscholar.org/api-docs/).
